# 🚜 PAVI Core: Ultra Production Pipeline V5
### YOLOv8/v11 + ByteTrack + Qwen 2.5-VL Auditor + SAM 2 + GPS Sync

---

**🆕 V5 New Features:**
- ✅ **Image-Only Mode** - Process folder of images (no video required)
- ✅ **GPS Optional** - Works without GPS data
- ✅ **Top-View Calibration** - GSD-based measurements for aerial/orthophoto images

**Pipeline Architecture:**
```
Video/Images + GPS Log → YOLO (v8/v11) → ByteTrack → Qwen VLM Audit → SAM 2 Segment
                                                         ↓              ↓
                                                    unmasked/       masked/
```

## 1️⃣ Install Dependencies

In [ ]:
# @title 🚀 Install & Setup
import os
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Install Core Dependencies
!pip install -q -U ultralytics
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
!pip install -q opencv-python pandas numpy tqdm scikit-image

# Install Qwen 2.5-VL (FREE local inference!)
!pip install -q transformers accelerate qwen-vl-utils

# Download SAM 2 Checkpoint
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

print("\n✅ Setup Complete.")

## 2️⃣ Configuration

In [ ]:
# @title ⚙️ Pipeline Configuration
import os
import math

# @markdown ### 🎯 Input Mode (NEW in V5!)
INPUT_MODE = "Video"  # @param ["Video", "Images"]
# @markdown ---

# @markdown ### 📂 Video Mode Paths
VIDEO_PATH = "/content/video.mp4"  # @param {type:"string"}
GPS_LOG_PATH = "/content/gps_log.csv"  # @param {type:"string"}

# @markdown ### 📁 Image Mode Paths (NEW in V5!)
IMAGES_FOLDER = "/content/images/"  # @param {type:"string"}
# @markdown *Upload folder containing .jpg/.png images*

# @markdown ### 🤖 Model Paths
DET_MODEL_PATH = "/content/best.pt"  # @param {type:"string"}
CALIB_IMAGE_PATH = "/content/calibration.jpg"  # @param {type:"string"}

# @markdown ### 🎯 Task Settings
TASK_TYPE = "Potholes"  # @param ["Potholes", "Cracks"]
INFERENCE_MODE = "Segmentation"  # @param ["Detection", "Segmentation"]

# @markdown ### 🧠 VLM Auditor (FREE - Qwen 2.5-VL)
VLM_VALIDATION = True  # @param {type:"boolean"}

# @markdown ### 📐 A4 Calibration (Priority 1)
MANUAL_CALIB_POINTS = []  # @param {type:"raw"}

# @markdown ### 📏 Top-View Calibration (NEW in V5!)
PIXELS_PER_METER = 500.0  # @param {type:"number"}
# @markdown *For top-view/aerial images: measure known distance in pixels*
USE_TOPVIEW_CALIBRATION = False  # @param {type:"boolean"}
# @markdown *Enable this for orthophoto/aerial images (uniform scale)*

# @markdown ### 📷 IPM Camera Parameters (Fallback)
CAMERA_HEIGHT_M = 1.2  # @param {type:"number"}
CAMERA_PITCH_DEG = 15.0  # @param {type:"number"}
CAMERA_FOV_DEG = 70.0  # @param {type:"number"}
ROAD_WIDTH_M = 3.5  # @param {type:"number"}

# @markdown ### ⚡ Performance Settings
FRAME_SKIP = 1  # @param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.25  # @param {type:"number"}
TRACK_BUFFER_SIZE = 15  # @param {type:"integer"}

print(f"✅ Mode: {INPUT_MODE}")
print(f"   Task: {TASK_TYPE} ({INFERENCE_MODE})")
print(f"   VLM: {'Enabled' if VLM_VALIDATION else 'Disabled'}")
print(f"   Top-View Calibration: {'Enabled' if USE_TOPVIEW_CALIBRATION else 'Disabled'}")

## 3️⃣ Interactive Calibration Tool 🖱️

In [ ]:
# @title 🖱️ Click A4 Corners (TL, TR, BR, BL)
# @markdown **Skip this if using Top-View Calibration**

import cv2
from base64 import b64encode
import numpy as np
from IPython.display import display, HTML
from google.colab import output

def show_interactive_calib(image_path):
    if not os.path.exists(image_path):
        print("⚠️ Calibration image not found.")
        return

    with open(image_path, "rb") as f:
        img_bytes = f.read()
    img_b64 = b64encode(img_bytes).decode('utf-8')
    
    html_code = f"""
    <div id="calib_container">
      <h3>🖱️ Click 4 A4 Corners: TL → TR → BR → BL</h3>
      <canvas id="calib_canvas" style="border:2px solid #0066ff; cursor:crosshair;"></canvas>
      <p id="points_log" style="font-family:monospace; background:#f0f0f0; padding:8px;">Points: []</p>
      <button onclick="resetPoints()" style="margin:5px; padding:8px 16px;">🔄 Reset</button>
      <button onclick="finishCalib()" style="margin:5px; padding:8px 16px; background:#0066ff; color:white;">✅ Done</button>
    </div>
    <script>
      var canvas = document.getElementById('calib_canvas');
      var ctx = canvas.getContext('2d');
      var img = new Image();
      img.src = "data:image/jpeg;base64,{img_b64}";
      var points = [];
      var scale = 1.0;
      
      img.onload = function() {{
        if (img.width > 900) {{ scale = 900 / img.width; }}
        canvas.width = img.width * scale;
        canvas.height = img.height * scale;
        redraw();
      }};
      
      function redraw() {{
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
        var labels = ['TL', 'TR', 'BR', 'BL'];
        for (var i = 0; i < points.length; i++) {{
          ctx.fillStyle = "#ff0000";
          ctx.beginPath();
          ctx.arc(points[i][0] * scale, points[i][1] * scale, 8, 0, 2*Math.PI);
          ctx.fill();
          ctx.fillStyle = "white";
          ctx.font = "bold 12px Arial";
          ctx.fillText(labels[i], points[i][0] * scale - 8, points[i][1] * scale + 4);
        }}
        if (points.length == 4) {{
          ctx.strokeStyle = "#00ff00";
          ctx.lineWidth = 2;
          ctx.beginPath();
          ctx.moveTo(points[0][0] * scale, points[0][1] * scale);
          for (var i = 1; i < 4; i++) {{
            ctx.lineTo(points[i][0] * scale, points[i][1] * scale);
          }}
          ctx.closePath();
          ctx.stroke();
        }}
        document.getElementById('points_log').innerText = "MANUAL_CALIB_POINTS = " + JSON.stringify(points);
      }}
      
      canvas.onclick = function(e) {{
        if (points.length >= 4) return;
        var rect = canvas.getBoundingClientRect();
        var x = (e.clientX - rect.left) / scale;
        var y = (e.clientY - rect.top) / scale;
        points.push([Math.round(x), Math.round(y)]);
        redraw();
      }};
      
      window.resetPoints = function() {{ points = []; redraw(); }};
      
      window.finishCalib = function() {{
        if (points.length != 4) {{ alert('Please click exactly 4 points!'); return; }}
        google.colab.kernel.invokeFunction('notebook.CalibCallback', [points], {{}});
      }};
    </script>
    """
    display(HTML(html_code))

def calib_callback(points):
    print("\n" + "="*60)
    print("✅ CALIBRATION COMPLETE! Copy this into Cell 2:")
    print("="*60)
    print(f"MANUAL_CALIB_POINTS = {points}")
    print("="*60)

output.register_callback('notebook.CalibCallback', calib_callback)

if os.path.exists(CALIB_IMAGE_PATH) and not USE_TOPVIEW_CALIBRATION:
    show_interactive_calib(CALIB_IMAGE_PATH)
elif USE_TOPVIEW_CALIBRATION:
    print(f"✅ Top-View mode enabled. Using PIXELS_PER_METER = {PIXELS_PER_METER}")
else:
    print("⚠️ No calibration image found. Using IPM fallback.")

## 4️⃣ Core Measurement Functions

In [ ]:
# @title 📐 Core Measurement Functions

import cv2
import numpy as np
import math
from skimage.morphology import skeletonize

def order_points(pts):
    """Order 4 points: TL, TR, BR, BL."""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def calculate_homography_and_scale(calibration_image_path, manual_points=None):
    """A4 Homography calculation."""
    if not calibration_image_path or not os.path.exists(calibration_image_path):
        return None, None, None
    
    img = cv2.imread(calibration_image_path)
    if img is None:
        return None, None, None
    
    rect = None
    
    if manual_points and len(manual_points) == 4:
        pts = np.array(manual_points, dtype="float32")
        rect = order_points(pts)
        print("✅ Using Manual A4 Calibration Points.")
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                       cv2.THRESH_BINARY, 11, 2)
        thresh = cv2.bitwise_not(thresh)
        cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
        
        for c in cnts:
            peri = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, 0.02 * peri, True)
            if len(approx) == 4:
                pts = approx.reshape(4, 2)
                rect = order_points(pts.astype("float32"))
                print("✅ Auto-Detected A4 Paper.")
                break
        
        if rect is None:
            print("⚠️ A4 auto-detection failed.")
            return None, None, None
    
    scale = 1000.0
    w_px = 0.210 * scale
    h_px = 0.297 * scale
    
    dst = np.array([[0, 0], [w_px, 0], [w_px, h_px], [0, h_px]], dtype="float32")
    H = cv2.getPerspectiveTransform(rect, dst)
    return H, scale, (int(w_px * 5), int(h_px * 5))

def get_local_scale(bbox, width, height, cam_height=1.2, pitch_deg=15, fov_deg=70):
    """Estimate pixels-per-meter at bbox location (perspective view)."""
    x1, y1, x2, y2 = map(int, bbox)
    bbox_center_y = (y1 + y2) / 2
    pitch = math.radians(pitch_deg)
    fov = math.radians(fov_deg)
    fx = (width / 2) / math.tan(fov / 2)
    cy = height / 2
    denom = (bbox_center_y - cy) * math.cos(pitch) + fx * math.sin(pitch)
    if denom < 1: denom = 1
    ground_distance = cam_height * fx / denom
    ground_distance = max(1.0, min(ground_distance, 20.0))
    view_width_m = 2 * ground_distance * math.tan(fov / 2)
    return width / view_width_m

def measure_defect_topview(mask, ppm, task_type, bbox=None):
    """
    Top-view measurement (uniform scale, no perspective correction).
    ppm = pixels per meter
    """
    # Clean mask inside bbox
    clean_mask = np.zeros_like(mask, dtype=bool)
    if bbox is not None:
        x1, y1, x2, y2 = map(int, bbox)
        clean_mask[y1:y2, x1:x2] = mask[y1:y2, x1:x2] > 0.0
    else:
        clean_mask = mask > 0.0
    
    if task_type == "Cracks":
        skeleton = skeletonize(clean_mask)
        pixel_length = np.sum(skeleton)
        
        if pixel_length == 0:
            contours, _ = cv2.findContours(
                clean_mask.astype(np.uint8) * 255, 
                cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            if contours:
                pixel_length = sum(cv2.arcLength(c, True) for c in contours) / 2
        
        length_m = pixel_length / ppm
        length_m = max(length_m, 0.01) if pixel_length > 0 else 0.0
        return length_m, (skeleton, clean_mask)
    else:
        pixel_area = np.sum(clean_mask)
        area_m2 = pixel_area / (ppm ** 2)
        area_m2 = max(area_m2, 0.001) if pixel_area > 0 else 0.0
        return area_m2, clean_mask

def measure_defect(mask, width, height, bbox, H, scale, bev_size, task_type):
    """Perspective-view measurement (A4 or IPM)."""
    clean_mask = np.zeros_like(mask, dtype=bool)
    if bbox is not None:
        x1, y1, x2, y2 = map(int, bbox)
        clean_mask[y1:y2, x1:x2] = mask[y1:y2, x1:x2] > 0.0
    else:
        clean_mask = mask > 0.0
    
    local_ppm = get_local_scale(bbox, width, height, CAMERA_HEIGHT_M, CAMERA_PITCH_DEG, CAMERA_FOV_DEG)
    
    if task_type == "Cracks":
        skeleton = skeletonize(clean_mask)
        pixel_length = np.sum(skeleton)
        
        if pixel_length == 0:
            contours, _ = cv2.findContours(
                clean_mask.astype(np.uint8) * 255, 
                cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            if contours:
                pixel_length = sum(cv2.arcLength(c, True) for c in contours) / 2
        
        if H is not None and scale is not None:
            val_m = pixel_length / scale
        else:
            val_m = pixel_length / local_ppm
        
        val_m = max(val_m, 0.01) if pixel_length > 0 else 0.0
        return val_m, (skeleton, clean_mask)
    
    else:
        pixel_area = np.sum(clean_mask)
        
        if H is not None and scale is not None:
            m_u8 = clean_mask.astype(np.uint8) * 255
            warped = cv2.warpPerspective(m_u8, H, bev_size)
            warped_area = cv2.countNonZero(warped)
            val_m = warped_area / (scale ** 2)
        else:
            val_m = pixel_area / (local_ppm ** 2)
        
        val_m = max(val_m, 0.001) if pixel_area > 0 else 0.0
        return val_m, clean_mask

print("✅ Measurement functions loaded.")
print("   - Top-View: Uniform scale (GSD-based)")
print("   - Perspective: A4 Homography or IPM fallback")

## 5️⃣ Load Qwen 2.5-VL (FREE VLM Auditor)

In [ ]:
# @title 🧠 Load Qwen 2.5-VL Model

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
import cv2
from PIL import Image

print("🔄 Loading Qwen 2.5-VL-3B...")

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

print(f"✅ Qwen 2.5-VL-3B Loaded!")

class QwenVLMAuditor:
    def __init__(self, model, processor):
        self.model = model
        self.processor = processor
        self.enabled = True
        
    def audit_defect(self, crop, defect_type):
        try:
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(crop_rgb)
            
            if defect_type.lower() == "cracks":
                prompt = '''Look at this road image. Could there be a CRACK here?
A crack is ANY visible line, fracture, or break pattern on the road surface.
Even if you are UNSURE, answer YES.
Only answer NO if CERTAIN this is a shadow, painted line, or normal texture.
Answer:'''
            else:
                prompt = '''Look at this road image. Could there be a POTHOLE here?
A pothole is ANY depression, hole, or damaged area on the road surface.
Even if you are UNSURE, answer YES.
Only answer NO if CERTAIN this is a shadow, manhole cover, or wet spot.
Answer:'''
            
            messages = [{"role": "user", "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": prompt}
            ]}]
            
            text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs,
                                    padding=True, return_tensors="pt").to(self.model.device)
            
            with torch.no_grad():
                generated_ids = self.model.generate(**inputs, max_new_tokens=10, do_sample=False)
            
            generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
            answer = self.processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0].strip().upper()
            
            return "NO" not in answer or "YES" in answer
        except Exception as e:
            print(f"⚠️ VLM Error: {e}")
            return True

auditor = QwenVLMAuditor(qwen_model, qwen_processor)
print("\n✅ VLM Auditor ready")

## 6️⃣ Main Pipeline Execution

In [ ]:
# @title 🚀 Run Pipeline
from ultralytics import YOLO
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from tqdm import tqdm
import pandas as pd
import shutil
import torch
import glob

MAX_CRACK_LENGTH_M = 3.0
MAX_POTHOLE_AREA_M2 = 1.5

def process_single_image(img_path, yolo, predictor, H_matrix, H_scale, bev_size, ppm_topview):
    """Process a single image (Image Mode)."""
    global auditor
    
    frame = cv2.imread(img_path)
    if frame is None:
        return []
    
    height, width = frame.shape[:2]
    filename = os.path.basename(img_path)
    
    # YOLO Detection (no tracking for images)
    results = yolo.predict(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
    
    if results.boxes is None or len(results.boxes) == 0:
        return []
    
    boxes = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy()
    
    records = []
    detection_idx = 0
    
    for box, conf in zip(boxes, confs):
        x1, y1, x2, y2 = map(int, box)
        crop = frame[y1:y2, x1:x2]
        
        # VLM Audit
        if VLM_VALIDATION:
            if not auditor.audit_defect(crop, TASK_TYPE.lower()):
                continue
        
        # Segmentation
        measurement = 0
        mask_data = None
        
        if INFERENCE_MODE == "Segmentation" and predictor:
            predictor.set_image(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            m, s, _ = predictor.predict(box=box, multimask_output=False)
            mask = m[0] if m.ndim == 3 else m
            
            if USE_TOPVIEW_CALIBRATION:
                measurement, mask_data = measure_defect_topview(mask, ppm_topview, TASK_TYPE, box)
            else:
                measurement, mask_data = measure_defect(mask, width, height, box, H_matrix, H_scale, bev_size, TASK_TYPE)
            
            if TASK_TYPE == "Cracks":
                measurement = min(measurement, MAX_CRACK_LENGTH_M)
            else:
                measurement = min(measurement, MAX_POTHOLE_AREA_M2)
        
        # Create label
        if TASK_TYPE == "Cracks":
            label = f"Crack | Length: {measurement:.2f}m"
        else:
            label = f"Pothole | Area: {measurement:.2f}m2"
        
        out_filename = f"{TASK_TYPE.lower()}_{detection_idx}_{filename}"
        
        # Save UNMASKED
        unmasked_frame = frame.copy()
        cv2.rectangle(unmasked_frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
        cv2.putText(unmasked_frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        cv2.imwrite(f"results/unmasked/{out_filename}", unmasked_frame)
        
        # Save MASKED
        if mask_data is not None:
            masked_frame = frame.copy()
            overlay = np.zeros_like(masked_frame)
            
            if TASK_TYPE == "Cracks":
                skeleton, clean_mask = mask_data
                overlay[clean_mask] = (0, 200, 200)
                overlay[skeleton] = (255, 255, 0)
            else:
                clean_mask = mask_data
                overlay[clean_mask] = (0, 0, 255)
            
            masked_frame = cv2.addWeighted(masked_frame, 0.7, overlay, 0.3, 0)
            cv2.rectangle(masked_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(masked_frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.imwrite(f"results/masked/{out_filename}", masked_frame)
        
        # Build record (no GPS for image mode)
        record = {
            'image_path': out_filename,
            'source_image': filename,
            'confidence': float(conf)
        }
        if TASK_TYPE == "Potholes":
            record['area_m2'] = round(measurement, 4)
        else:
            record['length_m'] = round(measurement, 4)
        
        records.append(record)
        detection_idx += 1
    
    return records

def run_image_pipeline():
    """Image-only pipeline (NEW in V5)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🔧 Device: {device}")
    print(f"📁 Images Folder: {IMAGES_FOLDER}")
    
    # Find images
    image_files = glob.glob(os.path.join(IMAGES_FOLDER, "*.jpg")) + \
                  glob.glob(os.path.join(IMAGES_FOLDER, "*.jpeg")) + \
                  glob.glob(os.path.join(IMAGES_FOLDER, "*.png"))
    
    if not image_files:
        print(f"❌ No images found in {IMAGES_FOLDER}")
        return pd.DataFrame()
    
    print(f"✅ Found {len(image_files)} images")
    
    # Load models
    yolo = YOLO(DET_MODEL_PATH)
    print(f"✅ YOLO Loaded: {DET_MODEL_PATH}")
    
    predictor = None
    if INFERENCE_MODE == "Segmentation":
        sam2 = build_sam2("sam2_hiera_s.yaml", "sam2_hiera_small.pt", device=device)
        predictor = SAM2ImagePredictor(sam2)
        print("✅ SAM 2 Loaded")
    
    # Calibration
    H_matrix, H_scale, bev_size = None, None, None
    ppm_topview = PIXELS_PER_METER
    
    if not USE_TOPVIEW_CALIBRATION:
        H_matrix, H_scale, bev_size = calculate_homography_and_scale(CALIB_IMAGE_PATH, MANUAL_CALIB_POINTS)
    else:
        print(f"📏 Top-View Calibration: {PIXELS_PER_METER} px/m")
    
    # Setup output
    os.makedirs("results/unmasked", exist_ok=True)
    os.makedirs("results/masked", exist_ok=True)
    
    all_records = []
    
    for img_path in tqdm(image_files, desc="Processing Images"):
        records = process_single_image(img_path, yolo, predictor, H_matrix, H_scale, bev_size, ppm_topview)
        all_records.extend(records)
    
    # Save results
    print("\n" + "="*60)
    print("📊 IMAGE PIPELINE COMPLETE")
    print("="*60)
    
    if all_records:
        df = pd.DataFrame(all_records)
        df.to_csv("results/detections.csv", index=False)
        
        print(f"✅ Detections: {len(df)}")
        print(f"   - Unmasked: {len(os.listdir('results/unmasked'))} files")
        print(f"   - Masked: {len(os.listdir('results/masked'))} files")
        
        shutil.make_archive("pipeline_results", 'zip', "results")
        print(f"\n📦 pipeline_results.zip created")
        
        try:
            from google.colab import files
            files.download("pipeline_results.zip")
        except:
            pass
    else:
        print("⚠️ No detections found.")
    
    return pd.DataFrame(all_records)

def run_video_pipeline():
    """Video pipeline (V4 logic)."""
    global auditor
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🔧 Device: {device}")
    
    yolo = YOLO(DET_MODEL_PATH)
    print(f"✅ YOLO Loaded: {DET_MODEL_PATH}")
    print(f"✅ VLM: {'Enabled' if VLM_VALIDATION else 'Disabled'}")
    
    predictor = None
    if INFERENCE_MODE == "Segmentation":
        sam2 = build_sam2("sam2_hiera_s.yaml", "sam2_hiera_small.pt", device=device)
        predictor = SAM2ImagePredictor(sam2)
        print("✅ SAM 2 Loaded")
    
    H_matrix, H_scale, bev_size = calculate_homography_and_scale(CALIB_IMAGE_PATH, MANUAL_CALIB_POINTS)
    if H_matrix is None:
        print(f"📐 Using IPM fallback")
    
    # GPS (optional)
    gps_df = None
    video_start_time = None
    if os.path.exists(GPS_LOG_PATH):
        try:
            gps_df = pd.read_csv(GPS_LOG_PATH)
            gps_df.columns = gps_df.columns.str.strip().str.lower()
            time_col = next((c for c in gps_df.columns if 'time' in c or 'date' in c), None)
            if time_col:
                gps_df[time_col] = pd.to_datetime(gps_df[time_col], format='mixed', utc=True)
                gps_df = gps_df.sort_values(time_col).rename(columns={time_col: 'time'})
                video_start_time = gps_df['time'].iloc[0]
                print(f"✅ GPS: {len(gps_df)} points")
        except Exception as e:
            print(f"⚠️ GPS Error: {e}")
    else:
        print("ℹ️ No GPS file - coordinates will be empty")
    
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width, height = int(cap.get(3)), int(cap.get(4))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    os.makedirs("results/unmasked", exist_ok=True)
    os.makedirs("results/masked", exist_ok=True)
    
    track_history = {}
    final_records = []
    validated_ids = set()
    rejected_ids = set()
    frame_idx = 0
    prev_active_ids = set()
    
    def process_track(tid, track_data):
        if len(track_data) == 0:
            return None
        
        best = max(track_data, key=lambda x: x['conf'])
        x1, y1, x2, y2 = map(int, best['box'])
        crop = best['frame'][y1:y2, x1:x2]
        
        if VLM_VALIDATION:
            if not auditor.audit_defect(crop, TASK_TYPE.lower()):
                rejected_ids.add(tid)
                return None
        
        validated_ids.add(tid)
        filename = f"{TASK_TYPE.lower()}_{tid}_frame{best['frame_idx']}.jpg"
        
        measurement = 0
        mask_data = None
        if INFERENCE_MODE == "Segmentation" and predictor:
            predictor.set_image(cv2.cvtColor(best['frame'], cv2.COLOR_BGR2RGB))
            m, s, _ = predictor.predict(box=best['box'], multimask_output=False)
            mask = m[0] if m.ndim == 3 else m
            measurement, mask_data = measure_defect(mask, width, height, best['box'], H_matrix, H_scale, bev_size, TASK_TYPE)
            
            if TASK_TYPE == "Cracks":
                measurement = min(measurement, MAX_CRACK_LENGTH_M)
            else:
                measurement = min(measurement, MAX_POTHOLE_AREA_M2)
        
        if TASK_TYPE == "Cracks":
            label = f"Crack | Length: {measurement:.2f}m"
        else:
            label = f"Pothole | Area: {measurement:.2f}m2"
        
        # Save UNMASKED
        unmasked_frame = best['frame'].copy()
        cv2.rectangle(unmasked_frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
        cv2.putText(unmasked_frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        cv2.imwrite(f"results/unmasked/{filename}", unmasked_frame)
        
        # Save MASKED
        if mask_data is not None:
            masked_frame = best['frame'].copy()
            overlay = np.zeros_like(masked_frame)
            if TASK_TYPE == "Cracks":
                skeleton, clean_mask = mask_data
                overlay[clean_mask] = (0, 200, 200)
                overlay[skeleton] = (255, 255, 0)
            else:
                clean_mask = mask_data
                overlay[clean_mask] = (0, 0, 255)
            masked_frame = cv2.addWeighted(masked_frame, 0.7, overlay, 0.3, 0)
            cv2.rectangle(masked_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(masked_frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.imwrite(f"results/masked/{filename}", masked_frame)
        
        record = {
            'track_id': tid,
            'confidence': float(best['conf']),
            'image_path': filename,
            'frame_idx': best['frame_idx']
        }
        if best['lat'] is not None:
            record['latitude'] = best['lat']
            record['longitude'] = best['lon']
        if TASK_TYPE == "Potholes":
            record['area_m2'] = round(measurement, 4)
        else:
            record['length_m'] = round(measurement, 4)
        
        return record
    
    pbar = tqdm(total=total_frames // FRAME_SKIP, desc=f"{TASK_TYPE} Detection")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx % FRAME_SKIP != 0:
            frame_idx += 1
            continue
        
        results = yolo.track(frame, persist=True, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
        current_active_ids = set()
        
        if results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            ids = results.boxes.id.cpu().numpy().astype(int)
            confs = results.boxes.conf.cpu().numpy()
            
            lat, lon = None, None
            if gps_df is not None and video_start_time is not None:
                curr_time = video_start_time + pd.to_timedelta(frame_idx / fps, unit='s')
                idx_near = (gps_df['time'] - curr_time).abs().idxmin()
                row = gps_df.loc[idx_near]
                lat, lon = float(row['latitude']), float(row['longitude'])
            
            for box, tid, conf in zip(boxes, ids, confs):
                current_active_ids.add(tid)
                if tid in rejected_ids or tid in validated_ids:
                    continue
                if tid not in track_history:
                    track_history[tid] = []
                track_history[tid].append({'box': box, 'frame': frame.copy(), 'conf': conf,
                                           'frame_idx': frame_idx, 'lat': lat, 'lon': lon})
                if len(track_history[tid]) >= TRACK_BUFFER_SIZE:
                    record = process_track(tid, track_history[tid])
                    if record:
                        final_records.append(record)
                    del track_history[tid]
        
        lost_ids = prev_active_ids - current_active_ids - validated_ids - rejected_ids
        for tid in lost_ids:
            if tid in track_history and len(track_history[tid]) >= 3:
                record = process_track(tid, track_history[tid])
                if record:
                    final_records.append(record)
            if tid in track_history:
                del track_history[tid]
        
        prev_active_ids = current_active_ids
        frame_idx += 1
        pbar.update(1)
    
    # Final flush
    for tid, track_data in list(track_history.items()):
        if tid not in validated_ids and tid not in rejected_ids and len(track_data) >= 3:
            record = process_track(tid, track_data)
            if record:
                final_records.append(record)
    
    cap.release()
    pbar.close()
    
    print("\n" + "="*60)
    print("📊 VIDEO PIPELINE COMPLETE")
    print("="*60)
    
    if final_records:
        df = pd.DataFrame(final_records)
        df.to_csv("results/detections.csv", index=False)
        print(f"✅ Detections: {len(df)}")
        print(f"   - Validated: {len(validated_ids)}")
        print(f"   - Rejected: {len(rejected_ids)}")
        shutil.make_archive("pipeline_results", 'zip', "results")
        print(f"\n📦 pipeline_results.zip created")
        try:
            from google.colab import files
            files.download("pipeline_results.zip")
        except:
            pass
    else:
        print("⚠️ No detections found.")
    
    return pd.DataFrame(final_records)

# Main execution
if INPUT_MODE == "Images":
    df = run_image_pipeline()
else:
    df = run_video_pipeline()

df.head()